Connected to Arena Python 3.13 (Python 3.13.13)

Connected to Arena Python 3.13 (Python 3.13.13)

In [ ]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import einops
import torch as t
import torchinfo
import wandb
from datasets import load_dataset
from einops.layers.torch import Rearrange
from jaxtyping import Float
import platform
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part5_vaes_and_gans"
#root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
root_dir = Path("/Users/sebastin/Documents/perso/ARENA_training/ARENA_3.0") if "QIMR" in platform.node() else Path("/home/sebastin/Documents/ARENA/ARENA_3.0") 
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

MAIN = __name__ == "__main__"

import part5_vaes_and_gans.tests as tests
import part5_vaes_and_gans.utils as utils
from plotly_utils import imshow

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

In [ ]:
celeb_data_dir = section_dir / "data/celeba"
celeb_image_dir = celeb_data_dir / "img_align_celeba"

os.makedirs(celeb_image_dir, exist_ok=True)

if len(list(celeb_image_dir.glob("*.jpg"))) > 0:
    print("Dataset already loaded.")
else:
    dataset = load_dataset("nielsr/CelebA-faces")
    print("Dataset loaded.")

    for idx, item in tqdm(enumerate(dataset["train"]), total=len(dataset["train"]), desc="Saving imgs...", ascii=True):
        # The image is already a JpegImageFile, so we can directly save it
        item["image"].save(celeb_image_dir / f"{idx:06}.jpg")

    print("All images have been saved.")

dataset_infos.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/462M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/463M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/463M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/202599 [00:00<?, ? examples/s]

Dataset loaded.


Saving imgs...: 100%|##########| 202599/202599 [06:03<00:00, 557.97it/s]

All images have been saved.


In [ ]:
def get_dataset(dataset: Literal["MNIST", "CELEB"], train: bool = True) -> Dataset:
    assert dataset in ["MNIST", "CELEB"]

    if dataset == "CELEB":
        image_size = 64
        assert train, "CelebA dataset only has a training set"
        transform = transforms.Compose(
            [
                transforms.Resize(image_size),
                transforms.CenterCrop(image_size),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )
        trainset = datasets.ImageFolder(root=exercises_dir / "part5_vaes_and_gans/data/celeba", transform=transform)

    elif dataset == "MNIST":
        img_size = 28
        transform = transforms.Compose(
            [
                transforms.Resize(img_size),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        trainset = datasets.MNIST(
            root=exercises_dir / "part5_vaes_and_gans/data",
            transform=transform,
            download=True,
            train=train,
        )

    return trainset

In [ ]:
def display_data(x: Tensor, nrows: int, title: str):
    """Displays a batch of data, using plotly."""
    ncols = x.shape[0] // nrows
    # Reshape into the right shape for plotting (make it 2D if image is monochrome)
    y = einops.rearrange(x, "(b1 b2) c h w -> (b1 h) (b2 w) c", b1=nrows).squeeze()
    # Normalize in the 0-1 range, then map to integer type
    y = (y - y.min()) / (y.max() - y.min())
    y = (y * 255).to(dtype=t.uint8)
    # Display data
    imshow(
        y,
        binary_string=(y.ndim == 2),
        height=50 * (nrows + 4),
        width=50 * (ncols + 5),
        title=f"{title}<br>single input shape = {x[0].shape}",
    )


trainset_mnist = get_dataset("MNIST")
trainset_celeb = get_dataset("CELEB")

# Display MNIST
x = next(iter(DataLoader(trainset_mnist, batch_size=25)))[0]
display_data(x, nrows=5, title="MNIST data")

# Display CelebA
x = next(iter(DataLoader(trainset_celeb, batch_size=25)))[0]
display_data(x, nrows=5, title="CelebA data")

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.86MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 136kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.10MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.5MB/s]


In [ ]:
testset = get_dataset("MNIST", train=False)
HOLDOUT_DATA = dict()
for data, target in DataLoader(testset, batch_size=1):
    if target.item() not in HOLDOUT_DATA:
        HOLDOUT_DATA[target.item()] = data.squeeze()
        if len(HOLDOUT_DATA) == 10:
            break
HOLDOUT_DATA = t.stack([HOLDOUT_DATA[i] for i in range(10)]).to(dtype=t.float, device=device).unsqueeze(1)

display_data(HOLDOUT_DATA, nrows=1, title="MNIST holdout data")

In [ ]:
from part2_cnns.solutions import BatchNorm2d, Conv2d, Linear, ReLU, Sequential
from part5_vaes_and_gans.solutions import ConvTranspose2d

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim_size: int, hidden_dim_size: int):
        """Creates the encoder & decoder modules."""
        super().__init__()
        self.encoder = Sequential(
            Conv2d(in_channels=1, out_channels=16, kernel_size=4, stride=2, padding=1),
            ReLU(),
            Conv2d(in_channels=16, out_channels=32, kernel_size=4, stride=2, padding=1),
            nn.Flatten(start_dim=1, end_dim=-1),
            Linear(in_features=32*7*7, out_features=hidden_dim_size, bias=True),
            ReLU(),
            Linear(in_features=hidden_dim_size, out_features=latent_dim_size, bias=True)
        )
        self.decoder = Sequential(
            Linear(in_features=latent_dim_size, out_features=hidden_dim_size, bias=True),
            ReLU(),
            Linear(in_features=hidden_dim_size, out_features=32*7*7, bias=True),
            ReLU(),
            Rearrange('b (c h w) -> b c h w', c=32, h=7, w=7),
            ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=4, stride=2, padding=1),
            ReLU(),
            ConvTranspose2d(in_channels=16, out_channels=1, kernel_size=4, stride=2, padding=1)
        )
        self.latent_dim_size = latent_dim_size
        self.hidden_dim_size = hidden_dim_size

    def forward(self, x: Tensor) -> Tensor:
        """Returns the reconstruction of the input, after mapping through encoder & decoder."""
        latent = self.encoder(x)
        return self.decoder(latent)


tests.test_autoencoder(Autoencoder)

All tests in `test_autoencoder` passed!


In [ ]:
class VAE(nn.Module):
    encoder: nn.Module
    decoder: nn.Module

    def __init__(self, latent_dim_size: int, hidden_dim_size: int):
        super().__init__()
        self.encoder = Sequential(
            Conv2d(in_channels=1, out_channels=16, kernel_size=4, stride=2, padding=1),
            ReLU(),
            Conv2d(in_channels=16, out_channels=32, kernel_size=4, stride=2, padding=1),
            nn.Flatten(start_dim=1, end_dim=-1),
            Linear(in_features=32*7*7, out_features=hidden_dim_size, bias=True),
            ReLU(),
            Linear(in_features=hidden_dim_size, out_features=2*latent_dim_size, bias=True),
            Rearrange('b (n latent_dim) -> n b latent_dim', n=2) 
        )
        self.decoder = Sequential(
            Linear(in_features=latent_dim_size, out_features=hidden_dim_size, bias=True),
            ReLU(),
            Linear(in_features=hidden_dim_size, out_features=32*7*7, bias=True),
            ReLU(),
            Rearrange('b (c h w) -> b c h w', c=32, h=7, w=7),
            ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=4, stride=2, padding=1),
            ReLU(),
            ConvTranspose2d(in_channels=16, out_channels=1, kernel_size=4, stride=2, padding=1)
        )
        self.latent_dim_size = latent_dim_size
        self.hidden_dim_size = hidden_dim_size

    def sample_latent_vector(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        """
        Passes `x` through the encoder, returns tuple of (sampled latent vector, mean, log std dev).
        This function can be used in `forward`, but also used on its own to generate samples for
        evaluation.
        """
        mu, logsigma = self.encoder(x) # shape (2, b, latent_dim_size)
        eps = t.randn_like(logsigma)
        latent = mu + t.exp(logsigma) * eps
        return latent, mu, logsigma

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        """
        Passes `x` through the encoder and decoder. Returns the reconstructed input, as well as mu
        and logsigma.
        """
        latent, mu, logsigma = self.sample_latent_vector(x)
        return self.decoder(latent), mu, logsigma
    



tests.test_vae(VAE)

All tests in `test_vae` passed!


In [ ]:
@dataclass
class AutoencoderArgs:
    # architecture
    latent_dim_size: int = 5
    hidden_dim_size: int = 128

    # data / training
    dataset: Literal["MNIST", "CELEB"] = "MNIST"
    batch_size: int = 512
    epochs: int = 10
    lr: float = 1e-3
    betas: tuple[float, float] = (0.5, 0.999)

    # logging
    use_wandb: bool = True
    wandb_project: str | None = "day5-autoencoder"
    wandb_name: str | None = None
    log_every_n_steps: int = 250

In [ ]:
@dataclass
class VAEArgs(AutoencoderArgs):
    wandb_project: str | None = "day5-vae-mnist"
    beta_kl: float = 0.1


class VAETrainer:
    def __init__(self, args: VAEArgs):
        self.args = args
        self.trainset = get_dataset(args.dataset)
        self.trainloader = DataLoader(self.trainset, batch_size=args.batch_size, shuffle=True, num_workers=8)
        self.model = VAE(
            latent_dim_size=args.latent_dim_size,
            hidden_dim_size=args.hidden_dim_size,
        ).to(device)
        self.optimizer = t.optim.Adam(self.model.parameters(), lr=args.lr, betas=args.betas)

    def training_step(self, img: Tensor):
        """
        Performs a training step on the batch of images in `img`. Returns the loss. Logs to wandb
        if enabled.
        """
        img = img.to(device)
        recon_img, mu, logsigma = self.model(img)
        D_KL = 0.5*(mu**2 + t.exp(2*logsigma) - 1) - logsigma
        loss = t.nn.functional.mse_loss(img, recon_img) + self.args.beta_kl * D_KL.mean() 
        
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        self.step += 1
        if self.args.use_wandb:
            if self.step % self.args.log_every_n_steps == 0:
                wandb.log({"train_loss": loss.item()}, step=self.step)
        return loss

    @t.inference_mode()
    def log_samples(self) -> None:
        """
        Evaluates model on holdout data, either logging to wandb or displaying output inline.
        """
        assert self.step > 0, "First call should come after a training step. Remember to increment `self.step`."
        output = self.model(HOLDOUT_DATA)[0]
        if self.args.use_wandb:
            output = (output - output.min()) / (output.max() - output.min())  # Normalize to [0, 1]
            output = (output * 255).to(dtype=t.uint8)  # Convert to uint8 for logging
            wandb.log({"images": [wandb.Image(arr) for arr in output.cpu().numpy()]}, step=self.step)
        else:
            display_data(t.concat([HOLDOUT_DATA, output]), nrows=2, title="VAE reconstructions")

    def train(self) -> VAE:
        """Performs a full training run."""
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name)
            wandb.watch(self.model)

        # YOUR CODE HERE - iterate over epochs, and train your model
        for epoch in range(self.args.epochs):
            pbar = tqdm(self.trainloader, desc=f"Epoch {epoch}/{self.args.epochs} - Training")
            mean_loss = 0
            for i, (img, label) in enumerate(pbar):
                loss = self.training_step(img)
                mean_loss += loss
                pbar.set_postfix(mean_loss=f"{mean_loss/(i+1):.3f}", n_img_seen=f"{self.step*self.args.batch_size}")

            self.log_samples()
            
        if self.args.use_wandb:
            wandb.finish()

        return self.model


args = VAEArgs(latent_dim_size=5, hidden_dim_size=100, use_wandb=False)
trainer = VAETrainer(args)
vae = trainer.train()

Epoch 0/10 - Training: 100%|██████████| 118/118 [00:04<00:00, 27.51it/s, mean_loss=0.665, n_img_seen=60416]


Epoch 1/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 51.69it/s, mean_loss=0.541, n_img_seen=120832]


Epoch 2/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 51.98it/s, mean_loss=0.519, n_img_seen=181248]


Epoch 3/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 52.58it/s, mean_loss=0.507, n_img_seen=241664]


Epoch 4/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 52.98it/s, mean_loss=0.498, n_img_seen=302080]


Epoch 5/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 52.29it/s, mean_loss=0.492, n_img_seen=362496]


Epoch 6/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 51.27it/s, mean_loss=0.487, n_img_seen=422912]


Epoch 7/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 51.72it/s, mean_loss=0.484, n_img_seen=483328]


Epoch 8/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 51.58it/s, mean_loss=0.481, n_img_seen=543744]


Epoch 9/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 51.55it/s, mean_loss=0.478, n_img_seen=604160]


In [ ]:
@dataclass
class VAEArgs(AutoencoderArgs):
    wandb_project: str | None = "day5-vae-mnist"
    beta_kl: float = 0.1


class VAETrainer:
    def __init__(self, args: VAEArgs):
        self.args = args
        self.trainset = get_dataset(args.dataset)
        self.trainloader = DataLoader(self.trainset, batch_size=args.batch_size, shuffle=True, num_workers=8)
        self.model = VAE(
            latent_dim_size=args.latent_dim_size,
            hidden_dim_size=args.hidden_dim_size,
        ).to(device)
        self.optimizer = t.optim.Adam(self.model.parameters(), lr=args.lr, betas=args.betas)

    def training_step(self, img: Tensor):
        """
        Performs a training step on the batch of images in `img`. Returns the loss. Logs to wandb
        if enabled.
        """
        img = img.to(device)
        recon_img, mu, logsigma = self.model(img)
        D_KL = 0.5*(mu**2 + t.exp(2*logsigma) - 1) - logsigma
        loss = t.nn.functional.mse_loss(img, recon_img) + self.args.beta_kl * D_KL.mean() 
        
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        self.step += 1
        if self.args.use_wandb:
            if self.step % self.args.log_every_n_steps == 0:
                wandb.log({"train_loss": loss.item()}, step=self.step)
        return loss

    @t.inference_mode()
    def log_samples(self) -> None:
        """
        Evaluates model on holdout data, either logging to wandb or displaying output inline.
        """
        assert self.step > 0, "First call should come after a training step. Remember to increment `self.step`."
        output = self.model(HOLDOUT_DATA)[0]
        if self.args.use_wandb:
            output = (output - output.min()) / (output.max() - output.min())  # Normalize to [0, 1]
            output = (output * 255).to(dtype=t.uint8)  # Convert to uint8 for logging
            wandb.log({"images": [wandb.Image(arr) for arr in output.cpu().numpy()]}, step=self.step)
        else:
            display_data(t.concat([HOLDOUT_DATA, output]), nrows=2, title="VAE reconstructions")

    def train(self) -> VAE:
        """Performs a full training run."""
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name)
            wandb.watch(self.model)

        # YOUR CODE HERE - iterate over epochs, and train your model
        for epoch in range(self.args.epochs):
            pbar = tqdm(self.trainloader, desc=f"Epoch {epoch}/{self.args.epochs} - Training")
            mean_loss = 0
            for i, (img, label) in enumerate(pbar):
                loss = self.training_step(img)
                mean_loss += loss
                pbar.set_postfix(mean_loss=f"{mean_loss/(i+1):.3f}", n_img_seen=f"{self.step*self.args.batch_size}")

            self.log_samples()
            
        if self.args.use_wandb:
            wandb.finish()

        return self.model


args = VAEArgs(latent_dim_size=5, hidden_dim_size=100, use_wandb=True)
trainer = VAETrainer(args)
vae = trainer.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/sebastin/.netrc.
wandb: Currently logged in as: sebnaze (sebnaze-qimr-berghofer-medical-research-institute) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 9/10 - Training: 100%|██████████| 118/118 [00:02<00:00, 51.84it/s, mean_loss=0.479, n_img_seen=604160]


train_loss,█▅▃▁
train_loss,0.4784


In [ ]:
def create_grid_of_latents(
    model, interpolation_range=(-1, 1), n_points=11, dims=(0, 1)
) -> Float[Tensor, "rows_x_cols latent_dims"]:
    """Create a tensor of zeros which varies along the 2 specified dimensions of the latent space."""
    grid_latent = t.zeros(n_points, n_points, model.latent_dim_size, device=device)
    x = t.linspace(*interpolation_range, n_points)
    grid_latent[..., dims[0]] = x.unsqueeze(-1)  # rows vary over dim=0
    grid_latent[..., dims[1]] = x  # cols vary over dim=1
    return grid_latent.flatten(0, 1)  # flatten over (rows, cols) into a single batch dimension

In [ ]:
grid_latent = create_grid_of_latents(vae, interpolation_range=(-1, 1))
output = vae.decoder(grid_latent)
utils.visualise_output(output, grid_latent, title="VAE latent space visualization")

In [ ]:
small_dataset = Subset(get_dataset("MNIST"), indices=range(0, 5000))
imgs = t.stack([img for img, label in small_dataset]).to(device)
labels = t.tensor([label for img, label in small_dataset]).to(device).int()

# We're getting the mean vector, which is the [0]-indexed output of the encoder
latent_vectors = vae.encoder(imgs)[0, :, :2]
holdout_latent_vectors = vae.encoder(HOLDOUT_DATA)[0, :, :2]

utils.visualise_input(latent_vectors.to('cpu'), labels.to('cpu'), holdout_latent_vectors.to('cpu'), HOLDOUT_DATA)

In [ ]:
#                           GANS
# -------------------------------------------------------------------   

class Tanh(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return (t.exp(2*x) - 1) / (t.exp(2*x) + 1)


class LeakyReLU(nn.Module):
    def __init__(self, negative_slope: float = 0.01):
        super().__init__()
        self.negative_slope = negative_slope

    def forward(self, x: Tensor) -> Tensor:
        return t.where(x >= 0, x, self.negative_slope * x)

    def extra_repr(self) -> str:
        return f"negative_slope={self.negative_slope}"


class Sigmoid(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return 1 / (1 + t.exp(-x))


tests.test_Tanh(Tanh)
tests.test_LeakyReLU(LeakyReLU)
tests.test_Sigmoid(Sigmoid)

All tests in `test_Tanh` passed.
All tests in `test_LeakyReLU` passed.
All tests in `test_Sigmoid` passed.


In [ ]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import einops
import torch as t
import torchinfo
import wandb
from datasets import load_dataset
from einops.layers.torch import Rearrange
from jaxtyping import Float
import platform
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part5_vaes_and_gans"
#root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
root_dir = Path("/Users/sebastin/Documents/perso/ARENA_training/ARENA_3.0") if "QIMR" in platform.node() else Path("/home/sebastin/Documents/ARENA/ARENA_3.0") 
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

MAIN = __name__ == "__main__"

import part5_vaes_and_gans.tests as tests
import part5_vaes_and_gans.utils as utils
from plotly_utils import imshow

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

In [ ]:
#                           GANS
# -------------------------------------------------------------------   

class Tanh(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return (t.exp(2*x) - 1) / (t.exp(2*x) + 1)


class LeakyReLU(nn.Module):
    def __init__(self, negative_slope: float = 0.01):
        super().__init__()
        self.negative_slope = negative_slope

    def forward(self, x: Tensor) -> Tensor:
        return t.where(x >= 0, x, self.negative_slope * x)

    def extra_repr(self) -> str:
        return f"negative_slope={self.negative_slope}"


class Sigmoid(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return 1 / (1 + t.exp(-x))


tests.test_Tanh(Tanh)
tests.test_LeakyReLU(LeakyReLU)
tests.test_Sigmoid(Sigmoid)

All tests in `test_Tanh` passed.
All tests in `test_LeakyReLU` passed.
All tests in `test_Sigmoid` passed.


In [ ]:
class Generator(nn.Module):
    def __init__(
        self,
        latent_dim_size: int = 100,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        """
        Implements the generator architecture from the DCGAN paper (the diagram at the top
        of page 4). We assume the size of the activations doubles at each layer (so image
        size has to be divisible by 2 ** len(hidden_channels)).

        Args:
            latent_dim_size:
                the size of the latent dimension, i.e. the input to the generator
            img_size:
                the size of the image, i.e. the output of the generator
            img_channels:
                the number of channels in the image (3 for RGB, 1 for grayscale)
            hidden_channels:
                the number of channels in the hidden layers of the generator (starting closest
                to the middle of the DCGAN and going outward, i.e. in chronological order for
                the generator)
        """
        n_layers = len(hidden_channels)
        assert img_size % (2**n_layers) == 0, "activation size must double at each layer"

        super().__init__()

        self.project_and_reshape = Sequential(
            Linear(in_features=latent_dim_size, out_features=hidden_channels[-1]*(img_size//(2**n_layers))**2, bias=False),
            Rearrange('b (c h w) -> b c h w', c=hidden_channels[-1], h=img_size//(2**n_layers), w=img_size//(2**n_layers)),
            BatchNorm2d(num_features=hidden_channels[-1]),
        )
        self.hidden_layers = Sequential(
            *[nn.Sequential(
                ConvTranspose2d(
                    in_channels=hidden_channels[-i-1],
                    out_channels=hidden_channels[-i-2],
                    kernel_size=4,
                    stride=2,
                    padding=1
                ),
                BatchNorm2d(num_features=hidden_channels[-i-2]),
                ReLU()
            ) for i in range(n_layers-1)],
            ConvTranspose2d(
                in_channels=hidden_channels[0],
                out_channels=img_channels,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            Tanh()
        )   

    def forward(self, x: Tensor) -> Tensor:
        x = self.project_and_reshape(x)
        x = self.hidden_layers(x)
        return x


class Discriminator(nn.Module):
    def __init__(
        self,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        """
        Implements the discriminator architecture from the DCGAN paper (the mirror image of
        the diagram at the top of page 4). We assume the size of the activations doubles at
        each layer (so image size has to be divisible by 2 ** len(hidden_channels)).

        Args:
            img_size:
                the size of the image, i.e. the input of the discriminator
            img_channels:
                the number of channels in the image (3 for RGB, 1 for grayscale)
            hidden_channels:
                the number of channels in the hidden layers of the discriminator (starting
                closest to the middle of the DCGAN and going outward, i.e. in reverse-
                chronological order for the discriminator)
        """
        n_layers = len(hidden_channels)
        assert img_size % (2**n_layers) == 0, "activation size must double at each layer"

        super().__init__()

        self.hidden_layers = Sequential(
            *[Sequential(
                Conv2d(
                    in_channels=img_channels,
                    out_channels=hidden_channels[0],
                    kernel_size=4,
                    stride=2,
                    padding=1
                ),
                LeakyReLU(negative_slope=0.2),
            ),
            *[nn.Sequential(
                Conv2d(
                    in_channels=hidden_channels[i],
                    out_channels=hidden_channels[i+1],
                    kernel_size=4,
                    stride=2,
                    padding=1
                ),
                BatchNorm2d(num_features=hidden_channels[i+1]),
                LeakyReLU(negative_slope=0.2)
            ) for i in range(n_layers-1)]
            ]
        )   
        
        self.classifier = Sequential(
            nn.Flatten(start_dim=1, end_dim=-1),
            Linear(in_features=hidden_channels[-1]*(img_size//(2**n_layers))**2, out_features=1, bias=False),
            Sigmoid()
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.hidden_layers(x)
        x = self.classifier(x)
        return x.squeeze()  # remove dummy `out_channels` dimension


class DCGAN(nn.Module):
    netD: Discriminator
    netG: Generator

    def __init__(
        self,
        latent_dim_size: int = 100,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        super().__init__()
        self.latent_dim_size = latent_dim_size
        self.img_size = img_size
        self.img_channels = img_channels
        self.hidden_channels = hidden_channels
        self.netD = Discriminator(img_size, img_channels, hidden_channels)
        self.netG = Generator(latent_dim_size, img_size, img_channels, hidden_channels)

In [ ]:
class Generator(nn.Module):
    def __init__(
        self,
        latent_dim_size: int = 100,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        """
        Implements the generator architecture from the DCGAN paper (the diagram at the top
        of page 4). We assume the size of the activations doubles at each layer (so image
        size has to be divisible by 2 ** len(hidden_channels)).

        Args:
            latent_dim_size:
                the size of the latent dimension, i.e. the input to the generator
            img_size:
                the size of the image, i.e. the output of the generator
            img_channels:
                the number of channels in the image (3 for RGB, 1 for grayscale)
            hidden_channels:
                the number of channels in the hidden layers of the generator (starting closest
                to the middle of the DCGAN and going outward, i.e. in chronological order for
                the generator)
        """
        n_layers = len(hidden_channels)
        assert img_size % (2**n_layers) == 0, "activation size must double at each layer"

        super().__init__()

        self.project_and_reshape = Sequential(
            Linear(in_features=latent_dim_size, out_features=hidden_channels[-1]*(img_size//(2**n_layers))**2, bias=False),
            Rearrange('b (c h w) -> b c h w', c=hidden_channels[-1], h=img_size//(2**n_layers), w=img_size//(2**n_layers)),
            BatchNorm2d(num_features=hidden_channels[-1]),
        )
        self.hidden_layers = nn.Sequential(
            *[nn.Sequential(
                ConvTranspose2d(
                    in_channels=hidden_channels[-i-1],
                    out_channels=hidden_channels[-i-2],
                    kernel_size=4,
                    stride=2,
                    padding=1
                ),
                BatchNorm2d(num_features=hidden_channels[-i-2]),
                ReLU()
            ) for i in range(n_layers-1)],
            ConvTranspose2d(
                in_channels=hidden_channels[0],
                out_channels=img_channels,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            Tanh()
        )   

    def forward(self, x: Tensor) -> Tensor:
        x = self.project_and_reshape(x)
        x = self.hidden_layers(x)
        return x


class Discriminator(nn.Module):
    def __init__(
        self,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        """
        Implements the discriminator architecture from the DCGAN paper (the mirror image of
        the diagram at the top of page 4). We assume the size of the activations doubles at
        each layer (so image size has to be divisible by 2 ** len(hidden_channels)).

        Args:
            img_size:
                the size of the image, i.e. the input of the discriminator
            img_channels:
                the number of channels in the image (3 for RGB, 1 for grayscale)
            hidden_channels:
                the number of channels in the hidden layers of the discriminator (starting
                closest to the middle of the DCGAN and going outward, i.e. in reverse-
                chronological order for the discriminator)
        """
        n_layers = len(hidden_channels)
        assert img_size % (2**n_layers) == 0, "activation size must double at each layer"

        super().__init__()

        self.hidden_layers = Sequential(
            *[nn.Sequential(
                Conv2d(
                    in_channels=img_channels,
                    out_channels=hidden_channels[0],
                    kernel_size=4,
                    stride=2,
                    padding=1
                ),
                LeakyReLU(negative_slope=0.2),
            ),
            *[nn.Sequential(
                Conv2d(
                    in_channels=hidden_channels[i],
                    out_channels=hidden_channels[i+1],
                    kernel_size=4,
                    stride=2,
                    padding=1
                ),
                BatchNorm2d(num_features=hidden_channels[i+1]),
                LeakyReLU(negative_slope=0.2)
            ) for i in range(n_layers-1)]
            ]
        )   
        
        self.classifier = Sequential(
            nn.Flatten(start_dim=1, end_dim=-1),
            Linear(in_features=hidden_channels[-1]*(img_size//(2**n_layers))**2, out_features=1, bias=False),
            Sigmoid()
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.hidden_layers(x)
        x = self.classifier(x)
        return x.squeeze()  # remove dummy `out_channels` dimension


class DCGAN(nn.Module):
    netD: Discriminator
    netG: Generator

    def __init__(
        self,
        latent_dim_size: int = 100,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        super().__init__()
        self.latent_dim_size = latent_dim_size
        self.img_size = img_size
        self.img_channels = img_channels
        self.hidden_channels = hidden_channels
        self.netD = Discriminator(img_size, img_channels, hidden_channels)
        self.netG = Generator(latent_dim_size, img_size, img_channels, hidden_channels)

In [ ]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import einops
import torch as t
import torchinfo
import wandb
from datasets import load_dataset
from einops.layers.torch import Rearrange
from jaxtyping import Float
import platform
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part5_vaes_and_gans"
#root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
root_dir = Path("/Users/sebastin/Documents/perso/ARENA_training/ARENA_3.0") if "QIMR" in platform.node() else Path("/home/sebastin/Documents/ARENA/ARENA_3.0") 
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

MAIN = __name__ == "__main__"

import part5_vaes_and_gans.tests as tests
import part5_vaes_and_gans.utils as utils
from part2_cnns.solutions import BatchNorm2d, Conv2d, Linear, ReLU, Sequential
from part5_vaes_and_gans.solutions import ConvTranspose2d
from plotly_utils import imshow

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

In [ ]:
from part2_cnns.utils import print_param_count
from part5_vaes_and_gans import solutions

print_param_count(Generator(), solutions.DCGAN().netG)
print_param_count(Discriminator(), solutions.DCGAN().netD)

Model 1, total params = 5906176
Model 2, total params = 5906176
All parameter shapes match, and are in the same order!


,name_1,shape_1,num_params_1,num_params_2,shape_2,name_2
0,project_and_reshape.0.weight,"(32768, 100)",3276800,3276800,"(32768, 100)",project_and_reshape.0.weight
1,project_and_reshape.2.weight,"(512,)",512,512,"(512,)",project_and_reshape.2.weight
2,project_and_reshape.2.bias,"(512,)",512,512,"(512,)",project_and_reshape.2.bias
3,hidden_layers.0.0.weight,"(512, 256, 4, 4)",2097152,2097152,"(512, 256, 4, 4)",hidden_layers.0.0.weight
4,hidden_layers.0.1.weight,"(256,)",256,256,"(256,)",hidden_layers.0.1.weight
5,hidden_layers.0.1.bias,"(256,)",256,256,"(256,)",hidden_layers.0.1.bias
6,hidden_layers.1.0.weight,"(256, 128, 4, 4)",524288,524288,"(256, 128, 4, 4)",hidden_layers.1.0.weight
7,hidden_layers.1.1.weight,"(128,)",128,128,"(128,)",hidden_layers.1.1.weight
8,hidden_layers.1.1.bias,"(128,)",128,128,"(128,)",hidden_layers.1.1.bias
9,hidden_layers.2.weight,"(128, 3, 4, 4)",6144,6144,"(128, 3, 4, 4)",hidden_layers.2.0.weight


Model 1, total params = 2661888
Model 2, total params = 2661888
All parameter shapes match, and are in the same order!


,name_1,shape_1,num_params_1,num_params_2,shape_2,name_2
0,hidden_layers.0.0.weight,"(128, 3, 4, 4)",6144,6144,"(128, 3, 4, 4)",hidden_layers.0.0.weight
1,hidden_layers.1.0.weight,"(256, 128, 4, 4)",524288,524288,"(256, 128, 4, 4)",hidden_layers.1.0.weight
2,hidden_layers.1.1.weight,"(256,)",256,256,"(256,)",hidden_layers.1.1.weight
3,hidden_layers.1.1.bias,"(256,)",256,256,"(256,)",hidden_layers.1.1.bias
4,hidden_layers.2.0.weight,"(512, 256, 4, 4)",2097152,2097152,"(512, 256, 4, 4)",hidden_layers.2.0.weight
5,hidden_layers.2.1.weight,"(512,)",512,512,"(512,)",hidden_layers.2.1.weight
6,hidden_layers.2.1.bias,"(512,)",512,512,"(512,)",hidden_layers.2.1.bias
7,classifier.1.weight,"(1, 32768)",32768,32768,"(1, 32768)",classifier.1.weight


In [ ]:
model = DCGAN().to(device)
x = t.randn(3, 100).to(device)
print(torchinfo.summary(model.netG, input_data=x), end="\n\n")
print(torchinfo.summary(model.netD, input_data=model.netG(x)))

Layer (type:depth-idx)                   Output Shape              Param #
Generator                                [3, 3, 64, 64]            --
├─Sequential: 1-1                        [3, 512, 8, 8]            --
│    └─Linear: 2-1                       [3, 32768]                3,276,800
│    └─Rearrange: 2-2                    [3, 512, 8, 8]            --
│    └─BatchNorm2d: 2-3                  [3, 512, 8, 8]            1,024
├─Sequential: 1-2                        [3, 3, 64, 64]            --
│    └─Sequential: 2-4                   [3, 256, 16, 16]          --
│    │    └─ConvTranspose2d: 3-1         [3, 256, 16, 16]          2,097,152
│    │    └─BatchNorm2d: 3-2             [3, 256, 16, 16]          512
│    │    └─ReLU: 3-3                    [3, 256, 16, 16]          --
│    └─Sequential: 2-5                   [3, 128, 32, 32]          --
│    │    └─ConvTranspose2d: 3-4         [3, 128, 32, 32]          524,288
│    │    └─BatchNorm2d: 3-5             [3, 128, 32, 32]     

In [ ]:
# Note that this should be done for all layers in both the generator and discriminator, 
# except for the output layer of the discriminator (the one with sigmoid activation), where weights and bias should be initialized to 0.  
def initialize_weights(model: nn.Module) -> None:
    """
    Initializes weights according to the DCGAN paper (details at the end of page 3 of the DCGAN
    paper), by modifying the weights of the model in place.
    """
    for m in model.modules():
        if isinstance(m, (Conv2d, ConvTranspose2d, Linear)):
            t.nn.init.normal_(m.weight.data, mean=0.0, std=0.02)
        elif isinstance(m, BatchNorm2d):
            t.nn.init.normal_(m.weight.data, mean=1.0, std=0.02)
            t.nn.init.zeros_(m.bias.data)


tests.test_initialize_weights(initialize_weights, ConvTranspose2d, Conv2d, Linear, BatchNorm2d)

All tests in `test_initialize_weights` passed.


In [ ]:
model = DCGAN().to(device)
x = t.randn(3, 100).to(device)
print(torchinfo.summary(model.netG, input_data=x), end="\n\n")
print(torchinfo.summary(model.netD, input_data=model.netG(x)))

Layer (type:depth-idx)                   Output Shape              Param #
Generator                                [3, 3, 64, 64]            --
├─Sequential: 1-1                        [3, 512, 8, 8]            --
│    └─Linear: 2-1                       [3, 32768]                3,276,800
│    └─Rearrange: 2-2                    [3, 512, 8, 8]            --
│    └─BatchNorm2d: 2-3                  [3, 512, 8, 8]            1,024
├─Sequential: 1-2                        [3, 3, 64, 64]            --
│    └─Sequential: 2-4                   [3, 256, 16, 16]          --
│    │    └─ConvTranspose2d: 3-1         [3, 256, 16, 16]          2,097,152
│    │    └─BatchNorm2d: 3-2             [3, 256, 16, 16]          512
│    │    └─ReLU: 3-3                    [3, 256, 16, 16]          --
│    └─Sequential: 2-5                   [3, 128, 32, 32]          --
│    │    └─ConvTranspose2d: 3-4         [3, 128, 32, 32]          524,288
│    │    └─BatchNorm2d: 3-5             [3, 128, 32, 32]     

In [ ]:
@dataclass
class DCGANArgs:
    """
    Class for the arguments to the DCGAN (training and architecture).
    Note, we use field(defaultfactory(...)) when our default value is a mutable object.
    """

    # architecture
    latent_dim_size: int = 100
    hidden_channels: list[int] = field(default_factory=lambda: [128, 256, 512])

    # data & training
    dataset: Literal["MNIST", "CELEB"] = "CELEB"
    batch_size: int = 64
    epochs: int = 3
    lr: float = 0.0002
    betas: tuple[float, float] = (0.5, 0.999)
    clip_grad_norm: float | None = 1.0

    # logging
    use_wandb: bool = False
    wandb_project: str | None = "day5-gan"
    wandb_name: str | None = None
    log_every_n_steps: int = 250


class DCGANTrainer:
    def __init__(self, args: DCGANArgs):
        self.args = args
        self.trainset = get_dataset(self.args.dataset)
        self.trainloader = DataLoader(self.trainset, batch_size=args.batch_size, shuffle=True, num_workers=8)

        batch, img_channels, img_height, img_width = next(iter(self.trainloader))[0].shape
        assert img_height == img_width

        self.model = DCGAN(args.latent_dim_size, img_height, img_channels, args.hidden_channels).to(device).train()
        self.optG = t.optim.Adam(self.model.netG.parameters(), lr=args.lr, betas=args.betas)
        self.optD = t.optim.Adam(self.model.netD.parameters(), lr=args.lr, betas=args.betas)

    def training_step_discriminator(
        self,
        img_real: Float[Tensor, "batch channels height width"],
        img_fake: Float[Tensor, "batch channels height width"],
    ) -> Float[Tensor, ""]:
        """
        Generates a real and fake image, and performs a gradient step on the discriminator to
        maximize log(D(x)) + log(1-D(G(z))). Logs to wandb if enabled.
        """
        disc_real = self.model.netD(img_real)
        disc_fake = self.model.netD(img_fake.detach())  # detach to avoid backprop to generator
        loss = -t.log(disc_real).mean() - t.log(1 - disc_fake).mean()
        loss.backward()
        if self.args.clip_grad_norm is not None:
            t.nn.utils.clip_grad_norm_(self.model.netD.parameters(), self.args.clip_grad_norm)
        self.optD.step()
        self.optD.zero_grad()
        if self.args.use_wandb:
            if self.step % self.args.log_every_n_steps == 0:
                wandb.log({"disc_loss": loss.item()}, step=self.step)
        return loss

    def training_step_generator(self, img_fake: Float[Tensor, "batch channels height width"]) -> Float[Tensor, ""]:
        """
        Performs a gradient step on the generator to maximize log(D(G(z))). Logs to wandb if enabled.
        """
        disc_fake = self.model.netD(img_fake)
        loss = -t.log(disc_fake).mean()
        loss.backward()
        if self.args.clip_grad_norm is not None:
            t.nn.utils.clip_grad_norm_(self.model.netG.parameters(), self.args.clip_grad_norm)
        self.optG.step()
        self.optG.zero_grad()
        if self.args.use_wandb:
            if self.step % self.args.log_every_n_steps == 0:
                wandb.log({"gen_loss": loss.item()}, step=self.step)
        return loss

    @t.inference_mode()
    def log_samples(self) -> None:
        """
        Performs evaluation by generating 8 instances of random noise and passing them through the
        generator, then optionally logging the results to Weights & Biases.
        """
        assert self.step > 0, "First call should come after a training step. Remember to increment `self.step`."
        self.model.netG.eval()

        # Generate random noise
        t.manual_seed(42)
        noise = t.randn(10, self.model.latent_dim_size).to(device)
        # Get generator output
        output = self.model.netG(noise)
        # Clip values to make the visualization clearer
        output = output.clamp(output.quantile(0.01), output.quantile(0.99))
        # Log to weights and biases
        if self.args.use_wandb:
            output = einops.rearrange(output, "b c h w -> b h w c").cpu().numpy()
            wandb.log({"images": [wandb.Image(arr) for arr in output]}, step=self.step)
        else:
            display_data(output, nrows=1, title="Generator-produced images")

        self.model.netG.train()

    def train(self) -> DCGAN:
        """Performs a full training run."""
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name)

        for epoch in range(self.args.epochs):
            progress_bar = tqdm(self.trainloader, total=len(self.trainloader), ascii=True, 
                                desc=f"Epoch {epoch}/{self.args.epochs} - Training - ")

            for img_real, label in progress_bar:
                # YOUR CODE HERE - fill in the training step for generator & discriminator
                img_real = img_real.to(device)
                noise = t.randn(img_real.size(0), self.model.latent_dim_size).to(device)
                img_fake = self.model.netG(noise)
                disc_loss = self.training_step_discriminator(img_real, img_fake)
                gen_loss = self.training_step_generator(img_fake)
                self.step += 1
                progress_bar.set_postfix(disc_loss=f"{disc_loss.item():.3f}", gen_loss=f"{gen_loss.item():.3f}", n_img_seen=f"{self.step*self.args.batch_size}")    
            self.log_samples()
            

        if self.args.use_wandb:
            wandb.finish()

        return self.model

In [ ]:
def get_dataset(dataset: Literal["MNIST", "CELEB"], train: bool = True) -> Dataset:
    assert dataset in ["MNIST", "CELEB"]

    if dataset == "CELEB":
        image_size = 64
        assert train, "CelebA dataset only has a training set"
        transform = transforms.Compose(
            [
                transforms.Resize(image_size),
                transforms.CenterCrop(image_size),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )
        trainset = datasets.ImageFolder(root=exercises_dir / "part5_vaes_and_gans/data/celeba", transform=transform)

    elif dataset == "MNIST":
        img_size = 28
        transform = transforms.Compose(
            [
                transforms.Resize(img_size),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        trainset = datasets.MNIST(
            root=exercises_dir / "part5_vaes_and_gans/data",
            transform=transform,
            download=True,
            train=train,
        )

    return trainset

In [ ]:
def display_data(x: Tensor, nrows: int, title: str):
    """Displays a batch of data, using plotly."""
    ncols = x.shape[0] // nrows
    # Reshape into the right shape for plotting (make it 2D if image is monochrome)
    y = einops.rearrange(x, "(b1 b2) c h w -> (b1 h) (b2 w) c", b1=nrows).squeeze()
    # Normalize in the 0-1 range, then map to integer type
    y = (y - y.min()) / (y.max() - y.min())
    y = (y * 255).to(dtype=t.uint8)
    # Display data
    imshow(
        y,
        binary_string=(y.ndim == 2),
        height=50 * (nrows + 4),
        width=50 * (ncols + 5),
        title=f"{title}<br>single input shape = {x[0].shape}",
    )


trainset_mnist = get_dataset("MNIST")
trainset_celeb = get_dataset("CELEB")

# Display MNIST
x = next(iter(DataLoader(trainset_mnist, batch_size=25)))[0]
display_data(x, nrows=5, title="MNIST data")

# Display CelebA
x = next(iter(DataLoader(trainset_celeb, batch_size=25)))[0]
display_data(x, nrows=5, title="CelebA data")

In [ ]:
testset = get_dataset("MNIST", train=False)
HOLDOUT_DATA = dict()
for data, target in DataLoader(testset, batch_size=1):
    if target.item() not in HOLDOUT_DATA:
        HOLDOUT_DATA[target.item()] = data.squeeze()
        if len(HOLDOUT_DATA) == 10:
            break
HOLDOUT_DATA = t.stack([HOLDOUT_DATA[i] for i in range(10)]).to(dtype=t.float, device=device).unsqueeze(1)

display_data(HOLDOUT_DATA, nrows=1, title="MNIST holdout data")

In [ ]:
# Arguments for MNIST
args = DCGANArgs(
    dataset="MNIST",
    hidden_channels=[12, 24],
    epochs=20,
    batch_size=128,
    use_wandb=False,
)
trainer = DCGANTrainer(args)
dcgan = trainer.train()

Epoch 0/20 - Training - : 100%|##########| 469/469 [00:06<00:00, 77.05it/s, disc_loss=0.830, gen_loss=0.678, n_img_seen=60032]


Epoch 1/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 84.94it/s, disc_loss=0.897, gen_loss=0.685, n_img_seen=120064]


Epoch 2/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 85.76it/s, disc_loss=0.876, gen_loss=0.626, n_img_seen=180096]


Epoch 3/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 84.48it/s, disc_loss=0.792, gen_loss=0.587, n_img_seen=240128]


Epoch 4/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 85.06it/s, disc_loss=0.799, gen_loss=0.679, n_img_seen=300160]


...

Epoch 15/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 82.76it/s, disc_loss=0.782, gen_loss=0.796, n_img_seen=960512]


Epoch 16/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 82.79it/s, disc_loss=0.711, gen_loss=0.626, n_img_seen=1020544]


Epoch 17/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 81.81it/s, disc_loss=nan, gen_loss=nan, n_img_seen=1080576]    


Epoch 18/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 81.54it/s, disc_loss=nan, gen_loss=nan, n_img_seen=1140608]


Epoch 19/20 - Training - : 100%|##########| 469/469 [00:05<00:00, 83.29it/s, disc_loss=nan, gen_loss=nan, n_img_seen=1200640]
